In [1]:
# install and import packages/libraries


!pip install pandas
!pip install unidecode
!pip install phonemizer
!pip install yt_dlp
!pip install musdb
!pip install demucs
!pip install "numpy>=1.26.0"
!pip install audiosr --no-deps
!pip install torchlibrosa ftfy progressbar2

import librosa
import numpy as np
import soundfile as sf
import librosa.display
import torch
import torch.nn as nn
%matplotlib inline
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from audiosr import super_resolution, build_model
import os
import pandas as pd
import yt_dlp
import subprocess
import glob
import musdb
import gc
import scipy.signal
from IPython.display import display

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.8/103.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.4/213.4 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 27.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.2/182.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.2/963.2 kB 14.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 9.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ..

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
# Define the AI algorithm structure using Pytorch's base model class
class SimpleAudioUNet(nn.Module):
    def __init__(self):
        super(SimpleAudioUNet, self).__init__()

        # Encoder
        self.enc1 = self.conv_block(1, 16) #finds 16 patterns in raw spectrogram
        self.enc2 = self.conv_block(16, 32) #combines the 16 patterns into 32 complex ones
        self.pool = nn.MaxPool2d(2) #halves image resolution to focus on broad features

        # Bottleneck
        self.bottleneck = self.conv_block(32, 64) #makes abstract representation of data

        # Decoder - processes data back to 16 patterns
        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec2 = self.conv_block(64, 32)
        self.up1 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.dec1 = self.conv_block(32, 16)

        self.final = nn.Conv2d(16, 1, kernel_size=1) #condenses 16 patterns into 1 output channel
        self.sigmoid = nn.Sigmoid() # creates the mask (all data between 0 and 1)

    def conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1), #looks for patterns
            nn.BatchNorm2d(out_ch), #normalises number (so they don't get huge)
            nn.ReLU(inplace=True), #filters out negative values (as algorithm can't understand them)
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1), #looks for patterns again
            nn.BatchNorm2d(out_ch), #normalises again
            nn.ReLU(inplace=True) #filters again
        )

    #actual path data takes when running the model
    def forward(self, x):
        e1 = self.enc1(x)  #high res features
        p1 = self.pool(e1) #reduce resolution
        e2 = self.enc2(p1) #mid res features
        p2 = self.pool(e2) #reduce resolution

        b = self.bottleneck(p2) #abstract representation of data

        d2 = self.up2(b) #increase resolution
        #check shapes match
        if d2.shape != e2.shape:
            d2 = torch.nn.functional.interpolate(d2, size=e2.shape[2:])
        d2 = torch.cat((d2, e2), dim=1) # Mix high res data with abstract data
        d2 = self.dec2(d2) #refines

        d1 = self.up1(d2) #increase res to original size
        # check shapes match
        if d1.shape != e1.shape:
            d1 = torch.nn.functional.interpolate(d1, size=e1.shape[2:])
        d1 = torch.cat((d1, e1), dim=1) # bring back sharp audio details
        d1 = self.dec1(d1) #refines again

        return self.sigmoid(self.final(d1)) #return the mask


In [3]:

# download tracks from MUSDB database
class MUSDBDataset(Dataset):
    def __init__(self, root_path="./musdb_7s", subset="train"):
        if not os.path.exists(root_path):
            os.makedirs(root_path)
        # scan the folder
        self.mus = musdb.DB(root=root_path, subsets=[subset], download=True)
        self.tracks = self.mus.tracks

    #return length
    def __len__(self):
        return len(self.tracks)

    def __getitem__(self, idx):
      track = self.tracks[idx]

      #make all tracks same length
      duration = 88200
      start = np.random.randint(0, len(track.audio) - duration)

      #pull mix and vocal tracks and convert to float tensors
      mix = torch.FloatTensor(track.audio[start : start+duration, 0])
      vocals = torch.FloatTensor(track.targets['vocals'].audio[start : start+duration, 0])

      #remove drum bleed from vocal track
      drum_audio = track.targets['drums'].audio[start:start+duration]
      drum_audio = np.mean(drum_audio, axis=1)
      if torch.is_tensor(vocals):
          vocals_np = vocals.detach().cpu().numpy()
      else:
          vocals_np = vocals
      vocal_cleaned = vocals_np - drum_audio
      vocal_cleaned = np.clip(vocal_cleaned, -1.0, 1.0) #normalisation
      vocals = torch.FloatTensor(vocal_cleaned.copy())


      return mix, vocals


In [4]:
# Train algorithm
def train_step(mixture_input, ground_vocal, model, optimizer):
    # clear memory
    optimizer.zero_grad()

    #convert 1D waveforms to 2D spectrograms
    mix_spec = torch.stft(mixture_input, n_fft=2048, hop_length=512, return_complex=True).abs()
    vocal_spec = torch.stft(ground_vocal, n_fft=2048, hop_length=512, return_complex=True).abs()
    mix_spec = mix_spec.unsqueeze(1)
    vocal_spec = vocal_spec.unsqueeze(1)

    # algorithm takes the mix and guesses the 0-1 mask
    predicted_mask = model(mix_spec)

    # apply the Mask (multiply the guess by the original mix)
    predicted_vocal = mix_spec * predicted_mask

    # use Mean Squared Error (MSE) to calculate error
    loss = torch.nn.functional.mse_loss(predicted_vocal, vocal_spec)

    # looks at its mistake and adjusts its fit
    loss.backward()
    optimizer.step()

    return loss.item() #returns error

In [5]:
# Make output file
def process_full_song(model, input_file, output_file, window_size=256):
    # Takes a long song, slices it, runs the algorithm on each bit, and stitches it back together
    model.eval() # stops running the training bit
    device = next(model.parameters()).device #check if using GPU or CPU

    # load the song, separate magnitude and phase
    y, sr = librosa.load(input_file, sr=44100)
    stft_full = librosa.stft(y)
    magnitude, phase = librosa.magphase(stft_full)

    # create array to hold output
    clean_mag_full = np.zeros_like(magnitude)

    # slicing loop
    num_frames = magnitude.shape[1]

    print(f"Processing {num_frames} frames...")

    with torch.no_grad(): # don't calculate gradients used for training to save memory
        for start in range(0, num_frames, window_size):
            end = start + window_size

            # pad with 0s if slice too small
            chunk = magnitude[:, start:end]
            actual_len = chunk.shape[1]
            if actual_len < window_size:
                chunk = np.pad(chunk, ((0,0), (0, window_size - actual_len))) #adds silence to fill

            # normalise
            max_val = np.max(chunk) + 1e-8
            chunk_norm = chunk / max_val

            # Convert to Tensor (1, 1, Freq, Time)
            chunk_tensor = torch.FloatTensor(chunk_norm[None, None, :, :]).to(device)

            # run the algorithm
            mask = model(chunk_tensor)

            # Apply mask and "un-normalise" back to original volume
            cleaned_chunk = (chunk_norm * mask.cpu().numpy().squeeze()) * max_val

            # Stitch back into full song file
            clean_mag_full[:, start:end] = cleaned_chunk[:, :actual_len]


    # reconstruct and Save
    reconstructed_file = "reconstruct.wav"
    print("Reconstructing audio...")
    y_out = reconstruct_audio(torch.FloatTensor(clean_mag_full), phase)
    sf.write(reconstructed_file, y_out, sr)

    # Move U-Net back to CPU and delete it
    model.cpu()
    del model
    #free up space
    torch.cuda.empty_cache()
    gc.collect()

    # use AudioSR to improve quality of audio - takes the isolated vocal and interpolates the missing high frequencies.
    # Load the pre-trained AudioSR model ('speech' is best for vocals)
    model_asr = build_model(model_name="speech", device="cuda" if torch.cuda.is_available() else "cpu")

    # ddim_steps=50 is the 'quality' setting - higher is better but slower.
    upscaled_audio = super_resolution(
        model_asr,
        input_file=reconstructed_file,
        seed=42,
        guidance_scale=3.5,
        ddim_steps=50
    )

    # Save the final 48kHz result
    # AudioSR returns a list - take the first element [0]
    final_vocal = upscaled_audio[0]
    #If it's a Torch Tensor, move it to CPU and NumPy
    if torch.is_tensor(final_vocal):
        final_vocal = final_vocal.detach().cpu().numpy()
    #remove extra dimension
    final_vocal = np.squeeze(final_vocal)
    sf.write(output_file, final_vocal, 48000)
    print(f"Final studio-quality vocal saved to: {output_file}")
    print(f"Success! Saved to {output_file}")



In [6]:
#reconstructs the audio file
def reconstruct_audio(predicted_vocal_magnitude, original_phase):
    # Move from Tensor back to NumPy for Librosa
    mag = predicted_vocal_magnitude.detach().cpu().numpy().squeeze()

    # multiply by phase and use inverse STFT (turn spectrogram back into sound wave)
    stft_reconstructed = mag * original_phase
    return librosa.istft(stft_reconstructed)

In [ ]:
def main():
    # setup model and what it is running on
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') #uses GPU if possible
    print(f"Using device: {device}")
    model = SimpleAudioUNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    #load dataset from MUSDB
    dataset = MUSDBDataset(root_path="./musdb_7s", subset="train")
    print(f"Successfully loaded {len(dataset)} tracks...")

    loader = DataLoader(dataset, batch_size=4, shuffle=True)

    print("Training model...")
    #train model
    model.train()
    for epoch in range(150):
        for i, (mix, vocal) in enumerate(loader):
            mix, vocal = mix.to(device), vocal.to(device)
            loss = train_step(mix, vocal, model, optimizer)
            if i % 5 == 0:
                print(f"Batch {i} | Loss: {loss:.6f}")



    #test
    # test_track = dataset.mus.tracks[2]
    # input_filename = "original.wav"
    # output_filename = "vocal_only.wav"
    # sf.write(input_filename, test_track.audio, test_track.rate)
    # print(f"Testing {input_filename}")

    input_filename = "gary.wav"
    output_filename = "vocal_only.wav"

    #output
    if os.path.exists(input_filename):
        process_full_song(model, input_filename, output_filename)


        try:
            #display spectogram
            # Load the audio file     'sr=None' is the native sampling rate
            y_in, sr_in = librosa.load(input_filename, sr=None)
            y_out, sr_out = librosa.load(output_filename, sr=None)

            # Compute the Short-Time Fourier Transform (STFT)
            # n_fft is the window size; hop_length is the slide distance
            stft_in = librosa.stft(y_in, n_fft=2048, hop_length=512)
            stft_out = librosa.stft(y_out, n_fft=2048, hop_length=512)

            # Convert amplitude to decibels (log scale) to make the visualisation much clearer
            spectrogram_in = librosa.amplitude_to_db(np.abs(stft_in), ref=np.max)
            spectrogram_out = librosa.amplitude_to_db(np.abs(stft_out), ref=np.max)

            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

            # Display the spectrograms
            img1 = librosa.display.specshow(spectrogram_in, sr=sr_in, hop_length=512, x_axis='time', y_axis='hz', ax=ax1)
            fig.colorbar(img1, ax=ax1, format='%+2.0f dB')
            ax1.set(title='Spectrogram of input (Hz)')


            # Plot Output
            img2 = librosa.display.specshow(spectrogram_out, sr=sr_in, hop_length=512, x_axis='time', y_axis='hz', ax=ax2)
            fig.colorbar(img2, ax=ax2, format='%+2.0f dB')
            ax2.set(title='Spectrogram of output (Hz)')
            ax2.set_xlim(0, 7)

            plt.tight_layout()

            display(fig)
            plt.show()

        except Exception as e:
            print(f"Error displaying spectrogram: {e}")


if __name__ == "__main__":
    main()

Using device: cuda


100%|██████████| 140M/140M [00:01<00:00, 134MB/s]


Successfully loaded 94 tracks...
Training model...


/usr/local/lib/python3.12/dist-packages/torch/functional.py:681: UserWarning: A window was not provided. A rectangular window will be applied,which is known to cause spectral leakage. Other windows such as torch.hann_window or torch.hamming_window are recommended to reduce spectral leakage.To suppress this warning and use a rectangular window, explicitly set `window=torch.ones(n_fft, device=<device>)`. (Triggered internally at /pytorch/aten/src/ATen/native/SpectralOps.cpp:835.)
  return _VF.stft(  # type: ignore[attr-defined]


Batch 0 | Loss: 11.409583
Batch 5 | Loss: 8.006813
Batch 10 | Loss: 10.215067
Batch 15 | Loss: 8.243659
Batch 20 | Loss: 12.114608
Batch 0 | Loss: 8.849442
Batch 5 | Loss: 7.526774
Batch 10 | Loss: 7.737128
Batch 15 | Loss: 5.764401
Batch 20 | Loss: 6.266544
Batch 0 | Loss: 6.943222
Batch 5 | Loss: 6.644758
Batch 10 | Loss: 4.747838
Batch 15 | Loss: 6.210443
Batch 20 | Loss: 12.997998
Batch 0 | Loss: 9.313196
Batch 5 | Loss: 8.451746
Batch 10 | Loss: 6.581535
Batch 15 | Loss: 5.300212
Batch 20 | Loss: 4.161572
Batch 0 | Loss: 8.262768
Batch 5 | Loss: 4.798720
Batch 10 | Loss: 8.805116
Batch 15 | Loss: 6.474886
Batch 20 | Loss: 5.658763
Batch 0 | Loss: 6.409910
Batch 5 | Loss: 5.659687
Batch 10 | Loss: 4.454604
Batch 15 | Loss: 5.045770
Batch 20 | Loss: 7.663525
Batch 0 | Loss: 3.763572
Batch 5 | Loss: 4.757282
Batch 10 | Loss: 4.200043
Batch 15 | Loss: 6.831546
Batch 20 | Loss: 6.616878
Batch 0 | Loss: 4.096449
Batch 5 | Loss: 5.296105
Batch 10 | Loss: 3.617486
Batch 15 | Loss: 7.60169